### Define the model and API to use. You need an API key here.

In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean
import json, re, ast

load_dotenv()

# This will look for "ANTHROPIC_API_KEY=" in a .env file.
# Go to Claude console, generate an API, and purchase some credits.
client = Anthropic()
model = "claude-haiku-4-5"

### Create the helper functions.

In [5]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 4096,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### Generate the dataset

In [33]:
# Function to generate a new dataset
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or Regex to complete.

Example output:
```json
[
    {
    "task": "Write a Python function get_ec2_instances_by_tag(tag_key: str, tag_value: str) -> list[dict] that returns a list of EC2 instance metadata dictionaries (containing 'InstanceId', 'InstanceType', and 'State') for all instances matching the given tag key-value pair.",
    "format": "python",
    "expected_output": "def get_ec2_instances_by_tag(tag_key: str, tag_value: str) -> list[dict]:\n    import boto3\n    ec2 = boto3.client('ec2')\n    response = ec2.describe_instances(\n        Filters=[{'Name': f'tag:{tag_key}', 'Values': [tag_value]}]\n    )\n    instances = []\n    for reservation in response.get('Reservations', []):\n        for instance in reservation.get('Instances', []):\n            instances.append({\n                'InstanceId': instance['InstanceId'],\n                'InstanceType': instance['InstanceType'],\n                'State': instance['State']['Name']\n            })\n    return instances",
    "solution_criteria": [
        "Defines a function named get_ec2_instances_by_tag",
        "Accepts two parameters: tag_key and tag_value, both typed as str",
        "Returns list[dict]",
        "Uses boto3.client('ec2')",
        "Calls describe_instances with a Filters argument using the tag:{tag_key} Name format",
        "Iterates over both Reservations and nested Instances in the response",
        "Each returned dict contains exactly 'InstanceId', 'InstanceType', and 'State'",
        "Extracts State as instance['State']['Name'], not the full State dict"
    ],
    "common_mistakes": [
        "Flattens only one level — iterates Reservations but forgets to iterate nested Instances",
        "Passes tag_key directly as the Filter Name instead of using the 'tag:{tag_key}' format",
        "Returns the full instance object instead of a trimmed metadata dict",
        "Extracts State as instance['State'] (the dict) rather than instance['State']['Name'] (the string)",
        "Hard-codes the tag key or value instead of using the parameters"
    ]
    },
    {
    "task": "Write a regex that matches AWS IAM ARNs for users, roles, or groups. The ARN format is: arn:aws:iam::<account-id>:<entity-type>/<name> where account-id is exactly 12 digits, entity-type is one of user, role, or group, and name contains only word characters (letters, digits, underscores), plus signs, equals signs, commas, periods, at signs, or hyphens, and is at least 1 character long.",
    "format": "regex",
    "expected_output": "^arn:aws:iam::\\d{12}:(user|role|group)\\/[\\w+=,.@-]+$",
    "solution_criteria": [
        "Pattern is anchored with ^ and $",
        "Matches the literal prefix arn:aws:iam::",
        "Account ID section uses \\d{12} (exactly 12 digits, not \\d+ or \\d{1,12})",
        "Entity type uses alternation (user|role|group) with no extra options",
        "A forward slash separates entity type from name",
        "Name character class includes word characters and all of: + = , . @ -",
        "Name quantifier is + (one or more), not * (zero or more)"
    ],
    "common_mistakes": [
        "Uses \\d+ instead of \\d{12}, allowing account IDs of any length",
        "Omits alternation and hard-codes only one entity type (e.g. role)",
        "Uses * instead of + for the name, allowing an empty name",
        "Missing one or more valid name characters from the character class (commonly = or +)",
        "Forgets to escape the forward slash in environments that require it"
    ]
    }
]
```

Generate exactly 3 JSON objects
* Exactly 1 python task
* Exactly 1 json task
* Exactly 1 regex task

"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
   # print(repr(text))
    return json.loads(text)

In [34]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

### Create the model grading pipeline.

In [35]:
# Function to grade a test case + output using a model

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

### Pass a test case to Claude

In [36]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

### Code grader

In [37]:
# Functions to validate the output structure

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


### Model grader and combine the Model and Code scores.

In [38]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [39]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [40]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.666666666666666


In [41]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport boto3\n\ndef list_s3_buckets_with_encryption():\n    s3_client = boto3.client('s3')\n    \n    response = s3_client.list_buckets()\n    buckets = response.get('Buckets', [])\n    \n    result = []\n    \n    for bucket in buckets:\n        bucket_name = bucket['Name']\n        encryption_enabled = False\n        \n        try:\n            encryption_response = s3_client.get_bucket_encryption(Bucket=bucket_name)\n            if 'ServerSideEncryptionConfiguration' in encryption_response:\n                rules = encryption_response['ServerSideEncryptionConfiguration'].get('Rules', [])\n                if rules and len(rules) > 0:\n                    encryption_enabled = True\n        except s3_client.exceptions.ServerSideEncryptionConfigurationNotFoundError:\n            encryption_enabled = False\n        except Exception:\n            encryption_enabled = False\n        \n        result.append({\n            'BucketName': bucket_name,\n            'Encry